# 1. Setup & Environment Configuration
Install required external packages, configure random seeds for reproducible experiments, and detect computational hardware.

In [13]:
import os
import sys
import random
import copy
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm

# Add project root to path for src imports
repo_root = os.path.abspath(os.path.join(os.path.dirname('__file__'), '../..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src.utils import seed_everything, load_config
seed_everything(2025)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device configured: {device} | Seed: {seed_val}')


Using device: cuda


# 2. Ontology Knowledge Graph Engine
Defines the `OntologyEngine` to extract lexical emotion signals, appraisal values, polarity, and intensity from an RDF Knowledge Graph.

In [14]:
# Initialize OntologyEngine using relative path to ontology RDF
from src.ontology_engine import OntologyEngine, VSFCOntologyEngine
from src.utils import load_config

config_path = os.path.join(repo_root, 'configs/vsmec.yaml')
config = load_config(config_path)
onto_cfg = config.get('ontology', {})

ontology_path = os.path.join(repo_root, onto_cfg.get('rdf_path', 'ontology/ekman_appraisal_ontology.rdf'))

ontengine = OntologyEngine(
    rdf_path=ontology_path,
    default_conf=onto_cfg.get('default_conf', 0.85),
    manual_boost=onto_cfg.get('manual_boost', 1.2),
    negation_attenuation=onto_cfg.get('negation_attenuation', 0.5),
    alpha_similarity=onto_cfg.get('alpha_similarity', 0.2),
    verbose=True,
)


Found ontology file: /kaggle/input/datasets/minkduck/kg-ekman6/ekman6_4_8.rdf
Loading Knowledge Graph: /kaggle/input/datasets/minkduck/kg-ekman6/ekman6_4_8.rdf
Successfully loaded 90420 triples.
Lexicon initialized with 2540 entries.


# 3. Ontology Diagnostics & Validation Suite
Validates the ontology feature extraction logic across key test cases (basic sentiments, soft negation, scope handling, and ambiguity filtering).

In [15]:
def decode_vector(vec, engine):
    """Decode a 24-dimensional feature vector into human-readable representations."""
    emo_vals = vec[:7]
    if np.max(emo_vals) > 0.01:
        top_idx = np.argmax(emo_vals)
        emotion = engine.ALLEMOTIONS[top_idx]
        score = emo_vals[top_idx]
    else:
        emotion = "Neutral/None"
        score = 0.0

    pol_vals = vec[19:22]
    pol_labels = ["Positive", "Negative", "Neutral"]
    if np.max(pol_vals) > 0:
        polarity = pol_labels[np.argmax(pol_vals)]
    else:
        polarity = "Unknown"

    has_neg = vec[22] > 0
    return emotion, score, polarity, has_neg


# Diagnostic test suite
test_cases = [
    ("món này ngon tuyệt", "Happiness", "Positive", "Basic Positive"),
    ("thất vọng tràn trề", "Sadness", "Negative", "Basic Negative"),
    (
        "tôi không thích nó",
        "Neutral/None",
        "Negative",
        "Soft Negation (Không thích)",
    ),
    ("chẳng vui chút nào", "Neutral/None", "Negative", "Soft Negation (Chẳng vui)"),
    (
        "tôi không hề thích",
        "Neutral/None",
        "Negative",
        "Scope Handling (Không hề...)",
    ),
    (
        "không phải là ghét",
        "Neutral/None",
        "Positive",
        "Double Negation (Không ghét)",
    ),
    ("làm hay nghỉ", "Neutral/None", "Unknown", "Disambiguation: Hay (Or)"),
    ("bún chả rất ngon", "Happiness", "Positive", "Disambiguation: Chả (Food)"),
    (
        "tôi là sinh viên",
        "Neutral/None",
        "Unknown",
        "Disambiguation: Pronouns/Stopwords",
    ),
]

print("==================== ONTOLOGY DIAGNOSTIC REPORT ====================")
results = []
pass_count = 0

for text, exp_emo, exp_pol, desc in test_cases:
    vec, _, trace = ontengine.getvector(text, debug=True)
    out_emo, out_score, out_pol, out_neg = decode_vector(vec, ontengine)

    emo_pass = (
        (out_score < 0.5 or out_emo == "Neutral")
        if exp_emo == "Neutral/None"
        else (out_emo == exp_emo and out_score > 0.3)
    )
    pol_pass = True if exp_pol == "Unknown" else (out_pol == exp_pol)

    status = "FAIL"
    if "Disambiguation: Chả" in desc and out_neg:
        status = "FAIL (False Negation)"
    elif "Soft Negation" in desc and out_pol == "Positive":
        status = "FAIL (Wrong Polarity)"
    elif emo_pass and pol_pass:
        status = "PASS"
        pass_count += 1

    matched_str = ", ".join([t["phrase"] for t in trace])
    results.append(
        {
            "Scenario": desc,
            "Input": text,
            "Matched": matched_str,
            "Prediction": f"{out_emo} ({out_score:.2f}) | {out_pol}",
            "NegFlag": "YES" if out_neg else "NO",
            "Status": status,
        }
    )

df_res = pd.DataFrame(results)
print(
    df_res[["Input", "Matched", "Prediction", "Status"]].to_string(index=False)
)
print("-" * 68)
print(f"Summary: {pass_count}/{len(test_cases)} test cases passed.")
print("====================================================================")

==================== ONTOLOGY DIAGNOSTIC REPORT ====================
             Input           Matched                    Prediction                Status
món này ngon tuyệt             tuyệt   Happiness (0.75) | Positive                  PASS
thất vọng tràn trề         thất vọng     Sadness (0.58) | Negative                  PASS
tôi không thích nó       không thích     Disgust (0.55) | Negative                  FAIL
chẳng vui chút nào               vui   Happiness (0.79) | Positive FAIL (Wrong Polarity)
tôi không hề thích      không, thích   Happiness (0.45) | Positive                  FAIL
không phải là ghét không, phải, ghét       Anger (0.17) | Negative                  FAIL
      làm hay nghỉ              nghỉ     Sadness (0.39) | Negative                  PASS
  bún chả rất ngon               rất   Happiness (0.42) | Positive                  PASS
  tôi là sinh viên                   Neutral/None (0.00) | Unknown                  PASS
-----------------------------------------

# 4. Dataset Preprocessing & DataLoaders
Loads the UIT-VSMEC dataset, encodes labels, extracts ontology vectors per sentence, and prepares PyTorch DataLoaders.

In [16]:
tokenizer = AutoTokenizer.from_pretrained("uitnlp/visobert")


class EmotionDataset(Dataset):

    def __init__(self, df, tokenizer, engine, max_len=128):
        self.df = df
        self.tokenizer = tokenizer
        self.engine = engine
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        txt = str(self.df.iloc[idx]["Sentence"])
        lbl = self.df.iloc[idx]["label_idx"]

        enc = self.tokenizer(
            txt,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        vec = self.engine.getvector(txt)
        return {
            "input_ids": enc["input_ids"].flatten(),
            "attention_mask": enc["attention_mask"].flatten(),
            "ontology_features": torch.tensor(vec, dtype=torch.float),
            "labels": torch.tensor(lbl, dtype=torch.long),
        }


# Dataset paths
BASE_DATA_PATH = "../../data/vsmec"

df_train = pd.read_excel(f"{BASE_DATA_PATH}/train_nor_811.xlsx")
df_valid = pd.read_excel(f"{BASE_DATA_PATH}/valid_nor_811.xlsx")
df_test = pd.read_excel(f"{BASE_DATA_PATH}/test_nor_811.xlsx")

# Drop null values
for df in [df_train, df_valid, df_test]:
    df.dropna(subset=["Sentence", "Emotion"], inplace=True)

# Encode emotion labels
labels = sorted(df_train["Emotion"].unique().astype(str))
le = LabelEncoder()
le.fit(labels)
print(f"Classes ({len(le.classes_)}): {le.classes_}")

for df in [df_train, df_valid, df_test]:
    df["label_idx"] = le.transform(df["Emotion"].astype(str))

# Create DataLoaders
BATCH_SIZE = 32
train_loader = DataLoader(
    EmotionDataset(df_train, tokenizer, ontengine), batch_size=BATCH_SIZE, shuffle=True
)
valid_loader = DataLoader(
    EmotionDataset(df_valid, tokenizer, ontengine), batch_size=BATCH_SIZE
)
test_loader = DataLoader(
    EmotionDataset(df_test, tokenizer, ontengine), batch_size=BATCH_SIZE
)

# Compute balanced class weights for loss computation
cls_w = compute_class_weight(
    "balanced",
    classes=np.unique(df_train["label_idx"]),
    y=df_train["label_idx"],
)
weights_tensor = torch.tensor(cls_w, dtype=torch.float).to(device)
print(f"Computed Class Weights: {weights_tensor}")

Classes (7): ['Anger' 'Disgust' 'Enjoyment' 'Fear' 'Other' 'Sadness' 'Surprise']
Computed Class Weights: tensor([2.0270, 0.7400, 0.5087, 2.4924, 0.7763, 0.8369, 3.2751],
       device='cuda:0')


# 5. Model Architectures
Defines neural-symbolic fusion architectures:
1. `ViSoBERT_Fusion`: Supports baseline, raw concatenation, and gated projection.
2. `ViSoBERT_DeepOntology`: Multi-layer bottleneck projection with residual gating.
3. `ViSoBERT_Ontology_CrossAttention`: Multi-head cross-attention querying discrete ontology concept tokens.

In [17]:
# Import proposed and benchmark model architectures from src.models
from src.models.final_comparison_models import (
    ViSoBERT_Baseline,
    ALDONAr_Model,
    KEAHT_Model,
    CombViSA_Model,
)

ONT_INPUT_DIM = 24
print('✅ Models loaded from src.models.final_comparison_models: ViSoBERT_Baseline, ALDONAr_Model, KEAHT_Model, CombViSA_Model')


# 6. Training & Evaluation Pipeline
Contains the training loop with early stopping, model checkpointing, and evaluation utilities.

In [18]:
def evaluate(model, loader):
    """Compute accuracy and macro F1 score on a given DataLoader."""
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in loader:
            ids, mask, ont, labels = [
                batch[k].to(device)
                for k in [
                    "input_ids",
                    "attention_mask",
                    "ontology_features",
                    "labels",
                ]
            ]
            outputs = model(ids, mask, ont)
            _, predicted = torch.max(outputs, 1)
            preds.extend(predicted.cpu().numpy())
            targets.extend(labels.cpu().numpy())
    return accuracy_score(targets, preds), f1_score(
        targets, preds, average="macro"
    )


def train_experiment(model, name, epochs=15, patience=4):
    """Train model with early stopping on validation macro F1 score."""
    print(f"\nTraining Model: {name}")
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    criterion = nn.CrossEntropyLoss(weight=weights_tensor)

    best_f1 = 0.0
    patience_counter = 0
    best_weights = copy.deepcopy(
        model.module.state_dict()
        if isinstance(model, nn.DataParallel)
        else model.state_dict()
    )

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for batch in train_loader:
            optimizer.zero_grad()
            ids, mask, ont, labels = [
                batch[k].to(device)
                for k in [
                    "input_ids",
                    "attention_mask",
                    "ontology_features",
                    "labels",
                ]
            ]
            loss = criterion(model(ids, mask, ont), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        val_acc, val_f1 = evaluate(model, valid_loader)
        print(
            f"Epoch {epoch + 1:02d} | Train Loss: {avg_loss:.4f} | Val F1: {val_f1:.4f} | Val Acc: {val_acc:.4f}",
            end=" ",
        )

        if val_f1 > best_f1:
            best_f1 = val_f1
            patience_counter = 0
            best_weights = copy.deepcopy(
                model.module.state_dict()
                if isinstance(model, nn.DataParallel)
                else model.state_dict()
            )
            print("(New best score)")
        else:
            patience_counter += 1
            print(f"(Patience: {patience_counter}/{patience})")

        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

    if isinstance(model, nn.DataParallel):
        model.module.load_state_dict(best_weights)
    else:
        model.load_state_dict(best_weights)
    return model

# 7. Model Training & Experimentation
Trains multiple model configurations:
- Baseline (ViSoBERT only)
- Raw Concat & Raw Gate
- Dense (64d) Concat & Dense (64d) Gate
- Residual Bottleneck Gate
- The ontology-guided cross-attention model.

In [19]:
NUM_CLASSES = len(le.classes_)

# 1. Baseline ViSoBERT
m_baseline = ViSoBERT_Baseline(NUM_CLASSES)
m_baseline = train_experiment(
    m_baseline, "1. Baseline (ViSoBERT)", epochs=15, patience=4
)

# 2. ALDONAr
m_aldonar = ALDONAr_Model(NUM_CLASSES, ont_dim=ONT_INPUT_DIM)
m_aldonar = train_experiment(
    m_aldonar, "2. ALDONAr (Dual-Gated Fusion)", epochs=15, patience=4
)

# 3. KEAHT
m_keaht = KEAHT_Model(NUM_CLASSES, ont_dim=ONT_INPUT_DIM, n_heads=8)
m_keaht = train_experiment(
    m_keaht,
    "3. KEAHT (Knowledge-Enriched Attention)",
    epochs=15,
    patience=4,
)

# 4. CombViSA
m_combvisa = CombViSA_Model(NUM_CLASSES, ont_dim=ONT_INPUT_DIM)
m_combvisa = train_experiment(
    m_combvisa,
    "4. CombViSA (ViSoBERT + BiLSTM + Ontology)",
    epochs=15,
    patience=4,
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: uitnlp/visobert
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Training Model: 1. Baseline (ViSoBERT)
Epoch 01 | Train Loss: 1.5901 | Val F1: 0.4700 | Val Acc: 0.4854 (New best score)
Epoch 02 | Train Loss: 1.0176 | Val F1: 0.5238 | Val Acc: 0.5292 (New best score)
Epoch 03 | Train Loss: 0.7257 | Val F1: 0.5317 | Val Acc: 0.5656 (New best score)
Epoch 04 | Train Loss: 0.4157 | Val F1: 0.5309 | Val Acc: 0.5583 (Patience: 1/4)
Epoch 05 | Train Loss: 0.2525 | Val F1: 0.5567 | Val Acc: 0.5816 (New best score)
Epoch 06 | Train Loss: 0.1462 | Val F1: 0.5592 | Val Acc: 0.5948 (New best score)
Epoch 07 | Train Loss: 0.0885 | Val F1: 0.5321 | Val Acc: 0.5656 (Patience: 1/4)
Epoch 08 | Train Loss: 0.0538 | Val F1: 0.5499 | Val Acc: 0.5904 (Patience: 2/4)
Epoch 09 | Train Loss: 0.0413 | Val F1: 0.5553 | Val Acc: 0.5962 (Patience: 3/4)
Epoch 10 | Train Loss: 0.0494 | Val F1: 0.5587 | Val Acc: 0.6137 (Patience: 4/4)
Early stopping triggered.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: uitnlp/visobert
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

XLMRobertaModel LOAD REPORT from: uitnlp/visobert
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Training Model: 3. KEAHT (Knowledge-Enriched Attention)
Epoch 01 | Train Loss: 1.5764 | Val F1: 0.4405 | Val Acc: 0.4461 (New best score)
Epoch 02 | Train Loss: 1.0476 | Val F1: 0.5019 | Val Acc: 0.5379 (New best score)
Epoch 03 | Train Loss: 0.6900 | Val F1: 0.5150 | Val Acc: 0.5510 (New best score)
Epoch 04 | Train Loss: 0.4097 | Val F1: 0.5218 | Val Acc: 0.5437 (New best score)
Epoch 05 | Train Loss: 0.2330 | Val F1: 0.4873 | Val Acc: 0.5335 (Patience: 1/4)
Epoch 06 | Train Loss: 0.1370 | Val F1: 0.5162 | Val Acc: 0.5714 (Patience: 2/4)
Epoch 07 | Train Loss: 0.0918 | Val F1: 0.5387 | Val Acc: 0.5845 (New best score)
Epoch 08 | Train Loss: 0.0503 | Val F1: 0.5303 | Val Acc: 0.5758 (Patience: 1/4)
Epoch 09 | Train Loss: 0.0425 | Val F1: 0.5150 | Val Acc: 0.5656 (Patience: 2/4)
Epoch 10 | Train Loss: 0.0437 | Val F1: 0.5188 | Val Acc: 0.5743 (Patience: 3/4)
Epoch 11 | Train Loss: 0.0354 | Val F1: 0.5052 | Val Acc: 0.5379 (Patience: 4/4)
Early stopping triggered.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: uitnlp/visobert
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Training Model: 4. CombViSA (ViSoBERT + BiLSTM + Ontology)
Epoch 01 | Train Loss: 1.8479 | Val F1: 0.3689 | Val Acc: 0.3848 (New best score)
Epoch 02 | Train Loss: 1.4066 | Val F1: 0.4948 | Val Acc: 0.5262 (New best score)
Epoch 03 | Train Loss: 0.9817 | Val F1: 0.5081 | Val Acc: 0.5569 (New best score)
Epoch 04 | Train Loss: 0.7189 | Val F1: 0.5270 | Val Acc: 0.5452 (New best score)
Epoch 05 | Train Loss: 0.5121 | Val F1: 0.5524 | Val Acc: 0.6020 (New best score)
Epoch 06 | Train Loss: 0.3609 | Val F1: 0.5340 | Val Acc: 0.5816 (Patience: 1/4)
Epoch 07 | Train Loss: 0.2682 | Val F1: 0.5423 | Val Acc: 0.5962 (Patience: 2/4)
Epoch 08 | Train Loss: 0.2044 | Val F1: 0.5303 | Val Acc: 0.5845 (Patience: 3/4)
Epoch 09 | Train Loss: 0.1868 | Val F1: 0.5100 | Val Acc: 0.5656 (Patience: 4/4)
Early stopping triggered.


# 8. Test Set Evaluation & Benchmark Comparison
Evaluates all trained variants on the test set and prints a comprehensive comparison table and classification metrics.

In [20]:
def get_predictions(model, loader):
    """Retrieve ground truth and model predictions on a dataset loader."""
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in loader:
            ids, mask, ont, labels = [
                batch[k].to(device)
                for k in [
                    "input_ids",
                    "attention_mask",
                    "ontology_features",
                    "labels",
                ]
            ]
            outputs = model(ids, mask, ont)
            _, pr = torch.max(outputs, 1)
            preds.extend(pr.cpu().numpy())
            targets.extend(labels.cpu().numpy())
    return targets, preds


# Collect predictions across models
y_true, p_base = get_predictions(m_baseline, test_loader)
_, p_aldonar = get_predictions(m_aldonar, test_loader)
_, p_keaht = get_predictions(m_keaht, test_loader)
_, p_comb = get_predictions(m_combvisa, test_loader)

models_preds = [
    ("Baseline (ViSoBERT)", p_base),
    ("ALDONAr", p_aldonar),
    ("KEAHT", p_keaht),
    ("CombViSA", p_comb),
]

summary_records = []
for name, preds in models_preds:
    summary_records.append(
        {
            "Model Architecture": name,
            "Accuracy": accuracy_score(y_true, preds),
            "Macro F1": f1_score(y_true, preds, average="macro"),
        }
    )

print("\n================ FINAL TEST BENCHMARK RESULTS ================")
df_benchmark = pd.DataFrame(summary_records)
print(df_benchmark.round(4).to_string(index=False))

print("\n========== CLASSIFICATION REPORT: ALDONAr ==========")
print(
    classification_report(y_true, p_aldonar, target_names=le.classes_, digits=4)
)

print("\n========== CLASSIFICATION REPORT: KEAHT ==========")
print(
    classification_report(y_true, p_keaht, target_names=le.classes_, digits=4)
)

print("\n========== CLASSIFICATION REPORT: CombViSA ==========")
print(
    classification_report(y_true, p_comb, target_names=le.classes_, digits=4)
)


================ FINAL TEST BENCHMARK RESULTS ================
 Model Architecture  Accuracy  Macro F1
Baseline (ViSoBERT)    0.5960    0.5592
            ALDONAr    0.5887    0.5698
              KEAHT    0.5887    0.5787
           CombViSA    0.6046    0.5869

========== CLASSIFICATION REPORT: ALDONAr ==========
              precision    recall  f1-score   support

       Anger     0.3922    0.5000    0.4396        40
     Disgust     0.5300    0.4015    0.4569       132
   Enjoyment     0.7349    0.6321    0.6797       193
        Fear     0.6034    0.7609    0.6731        46
       Other     0.4821    0.6279    0.5455       129
     Sadness     0.7130    0.6638    0.6875       116
    Surprise     0.4762    0.5405    0.5063        37

    accuracy                         0.5887       693
   macro avg     0.5617    0.5895    0.5698       693
weighted avg     0.6028    0.5887    0.5900       693


========== CLASSIFICATION REPORT: KEAHT ==========
              precision    recall

# 9. Neuro-Symbolic Rule-Based Reasoner
Applies post-processing domain rules from the ontology to override low-confidence neural model predictions.

In [21]:
class NeuroSymbolicReasonerVSMEC:
    """Post-processing override module combining neural predictions with ontology rules."""

    def __init__(
        self,
        model,
        engine,
        tokenizer,
        device,
        ontology_threshold=0.4,
        model_conf_limit=0.65,
    ):
        self.model = model.module if isinstance(model, nn.DataParallel) else model
        self.engine = engine
        self.tokenizer = tokenizer
        self.device = device
        self.ont_threshold = ontology_threshold
        self.model_conf_limit = model_conf_limit
        self.idx2label = {i: label for i, label in enumerate(le.classes_)}

    def infer(self, text):
        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=128,
            padding="max_length",
        )
        input_ids = inputs["input_ids"].to(self.device)
        attention_mask = inputs["attention_mask"].to(self.device)

        vec, _, _ = self.engine.getvector(text, debug=True)
        ont_tensor = torch.tensor(np.array([vec]), dtype=torch.float).to(
            self.device
        )

        self.model.eval()
        with torch.no_grad():
            outputs = self.model(input_ids, attention_mask, ont_tensor)
            probs = F.softmax(outputs, dim=1)

        pred_idx = torch.argmax(probs).item()
        confidence = probs[0][pred_idx].item()
        neural_label = self.idx2label[pred_idx]

        # Symbolic override logic
        ont_emotions = vec[0:7]
        max_ont_idx = np.argmax(ont_emotions)
        max_ont_score = ont_emotions[max_ont_idx]
        ont_label_name = self.engine.ALLEMOTIONS[max_ont_idx]

        ont_to_vsmec = {"Happiness": "Enjoyment", "Neutral": "Other"}
        final_ont_label = ont_to_vsmec.get(ont_label_name, ont_label_name)
        final_label = neural_label

        if neural_label == "Other" or confidence < self.model_conf_limit:
            if (
                max_ont_score > self.ont_threshold
                and final_ont_label in le.classes_
            ):
                final_label = final_ont_label

        return neural_label, final_label


reasoner_vsmec = NeuroSymbolicReasonerVSMEC(
    model=m_baseline,
    engine=ontengine,
    tokenizer=tokenizer,
    device=device,
    ontology_threshold=0.4,
    model_conf_limit=0.65,
)

y_true_ns = []
y_pred_base = []
y_pred_neuro = []

for _, row in df_test.iterrows():
    text = row["Sentence"]
    true_label_str = row["Emotion"]

    base_lbl, neuro_lbl = reasoner_vsmec.infer(text)
    y_true_ns.append(true_label_str)
    y_pred_base.append(base_lbl)
    y_pred_neuro.append(neuro_lbl)

acc_base = accuracy_score(y_true_ns, y_pred_base)
f1_base = f1_score(y_true_ns, y_pred_base, average="macro")
acc_neuro = accuracy_score(y_true_ns, y_pred_neuro)
f1_neuro = f1_score(y_true_ns, y_pred_neuro, average="macro")

df_rule_res = pd.DataFrame(
    {
        "Metric": ["Accuracy", "Macro F1"],
        "Baseline (ViSoBERT)": [acc_base, f1_base],
        "Baseline + Ontology Rules": [acc_neuro, f1_neuro],
        "Delta": [acc_neuro - acc_base, f1_neuro - f1_base],
    }
)

print("\n================ RULE-BASED POST-PROCESSING EVALUATION ================")
print(df_rule_res.round(4).to_string(index=False))

overridden_count = sum(
    1 for b, n in zip(y_pred_base, y_pred_neuro) if b != n
)
print(
    f"\nTotal overridden samples: {overridden_count}/{len(df_test)} ({overridden_count / len(df_test) * 100:.2f}%)"
)


================ RULE-BASED POST-PROCESSING EVALUATION ================
  Metric  Baseline (ViSoBERT)  Baseline + Ontology Rules   Delta
Accuracy               0.5960                     0.6003  0.0043
Macro F1               0.5592                     0.5591 -0.0001

Total overridden samples: 56/693 (8.08%)


# 10. Statistical Significance Testing (Paired Permutation Test)
Performs non-parametric paired permutation tests (10,000 iterations) to determine if improvements are statistically significant ($p < 0.05$).

In [22]:
from src.significance_test import paired_permutation_test, macro_f1


comparisons = [
    ("Baseline vs ALDONAr", p_base, p_aldonar),
    ("Baseline vs KEAHT", p_base, p_keaht),
    ("Baseline vs CombViSA", p_base, p_comb),
    ("ALDONAr vs KEAHT", p_aldonar, p_keaht),
    ("KEAHT vs CombViSA", p_keaht, p_comb),
]

print(
    "\n============== STATISTICAL SIGNIFICANCE TESTS (10,000 ITERATIONS) =============="
)

for name, preds_a, preds_b in comparisons:
    f1_a, f1_b, diff_f1, p_val_f1 = paired_permutation_test(
        y_true, preds_a, preds_b, metric_func=macro_f1
    )
    acc_a, acc_b, diff_acc, p_val_acc = paired_permutation_test(
        y_true, preds_a, preds_b, metric_func=accuracy_score
    )

    sig_f1 = "Significant (p < 0.05)" if p_val_f1 < 0.05 else "Not Significant"
    sig_acc = (
        "Significant (p < 0.05)" if p_val_acc < 0.05 else "Not Significant"
    )

    print(f"\nComparison: {name}")
    print(
        f"  - Macro F1: {f1_a:.4f} vs {f1_b:.4f} | Diff: {diff_f1:.4f} | p-value: {p_val_f1:.5f} ({sig_f1})"
    )
    print(
        f"  - Accuracy: {acc_a:.4f} vs {acc_b:.4f} | Diff: {diff_acc:.4f} | p-value: {p_val_acc:.5f} ({sig_acc})"
    )

print(
    "\n================================================================================"
)


============== STATISTICAL SIGNIFICANCE TESTS (10,000 ITERATIONS) ==============

Comparison: Baseline vs ALDONAr
  - Macro F1: 0.5592 vs 0.5698 | Diff: 0.0105 | p-value: 0.60024 (Not Significant)
  - Accuracy: 0.5960 vs 0.5887 | Diff: 0.0072 | p-value: 0.75372 (Not Significant)

Comparison: Baseline vs KEAHT
  - Macro F1: 0.5592 vs 0.5787 | Diff: 0.0194 | p-value: 0.26317 (Not Significant)
  - Accuracy: 0.5960 vs 0.5887 | Diff: 0.0072 | p-value: 0.70613 (Not Significant)

Comparison: Baseline vs CombViSA
  - Macro F1: 0.5592 vs 0.5869 | Diff: 0.0276 | p-value: 0.16508 (Not Significant)
  - Accuracy: 0.5960 vs 0.6046 | Diff: 0.0087 | p-value: 0.68253 (Not Significant)

Comparison: ALDONAr vs KEAHT
  - Macro F1: 0.5698 vs 0.5787 | Diff: 0.0089 | p-value: 0.63604 (Not Significant)
  - Accuracy: 0.5887 vs 0.5887 | Diff: 0.0000 | p-value: 1.00000 (Not Significant)

Comparison: KEAHT vs CombViSA
  - Macro F1: 0.5787 vs 0.5869 | Diff: 0.0082 | p-value: 0.66253 (Not Significant)
  - Accuracy